# Per-Bin Model Comparison

Tests all model types on each wallclock bin independently to determine
which model works best for each job population.

**Models:** Random Forest, XGBoost Adjusted, XGBoost DART, LightGBM, TF-IDF kNN, Baseline
**Bins:** <=2h, 2-4h, 4-24h, 24-48h, >48h
**Dataset:** NLR Kestrel, expanded window (~193 days, 2.7M rows)
**Lookback:** 120 days, 120 windows × 6h

## 1. Setup

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from hpc_oda_commons.models.job_runtime_xgboost.model import (
    JobRuntimeXGBoostConfig, JobRuntimeXGBoostModel,
)
from hpc_oda_commons.models.job_runtime_random_forest.model import (
    JobRuntimeRandomForestConfig, JobRuntimeRandomForestModel,
)
from hpc_oda_commons.models.job_runtime_tfidf_knn.model import (
    JobRuntimeTfidfKnnConfig, JobRuntimeTfidfKnnModel,
)
from hpc_oda_commons.models.experimental.lightgbm_model import (
    ExperimentalLightGBMConfig, ExperimentalLightGBMModel,
)
from hpc_oda_commons.models.experimental.xgboost_dart_model import (
    ExperimentalXGBoostDartConfig, ExperimentalXGBoostDartModel,
)
from hpc_oda_commons.models.experimental.xgboost_adjusted_model import (
    ExperimentalXGBoostAdjustedConfig, ExperimentalXGBoostAdjustedModel,
)
from hpc_oda_commons.benchmark.runner import run_rolling_baseline

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'

In [ ]:
# Load expanded window
table = pq.read_table(DATA_PATH)
lo = datetime(2025, 1, 1, tzinfo=timezone.utc)
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()
rows_all = df.to_dict('records')
print(f'Loaded: {len(rows_all):,} rows')
print(f'Span: {(df["submit_time"].max() - df["submit_time"].min()).days} days')

## 2. Split into wallclock bins

In [ ]:
BIN_EDGES_H = [0, 2, 4, 24, 48, float('inf')]
BIN_LABELS = ['<=2h', '2-4h', '4-24h', '24-48h', '>48h']

def assign_bin(row):
    wc_h = (row.get('requested_seconds') or 0) / 3600
    for i in range(len(BIN_EDGES_H) - 1):
        if wc_h <= BIN_EDGES_H[i + 1]:
            return BIN_LABELS[i]
    return BIN_LABELS[-1]

bins = {label: [] for label in BIN_LABELS}
for row in rows_all:
    bins[assign_bin(row)].append(row)

print(f'{"Bin":<10} {"Jobs":>10} {"% of total":>10}')
print('-' * 35)
for label in BIN_LABELS:
    print(f'{label:<10} {len(bins[label]):>10,} {len(bins[label])/len(rows_all)*100:>9.1f}%')

## 3. Define models

In [ ]:
SHARED = dict(
    n_windows=120,
    test_window_hours=6,
    training_lookback_days=120,
    max_svd_components=256,
    target_max_one_hot_width=2048,
    random_state=42,
)

MODELS = {
    'Random Forest': lambda: JobRuntimeRandomForestModel(
        JobRuntimeRandomForestConfig(**SHARED, n_estimators=100, max_depth=16)),
    'XGBoost Adjusted': lambda: ExperimentalXGBoostAdjustedModel(
        ExperimentalXGBoostAdjustedConfig(**SHARED, n_estimators=200, max_depth=12,
            learning_rate=0.03, min_child_weight=5, gamma=0.1)),
    'XGBoost DART': lambda: ExperimentalXGBoostDartModel(
        ExperimentalXGBoostDartConfig(**SHARED, rate_drop=0.1, skip_drop=0.5)),
    'LightGBM': lambda: ExperimentalLightGBMModel(
        ExperimentalLightGBMConfig(**SHARED)),
    'TF-IDF kNN': lambda: JobRuntimeTfidfKnnModel(
        JobRuntimeTfidfKnnConfig(n_windows=SHARED['n_windows'],
            test_window_hours=SHARED['test_window_hours'],
            training_lookback_days=SHARED['training_lookback_days'],
            k=5, n_hash_features=2**14)),
}

print(f'Models: {list(MODELS.keys())}')

## 4. Run all models on each bin

In [ ]:
results = {}  # (bin, model) -> {mae, rmse, scored}

for bin_label in BIN_LABELS:
    bin_rows = bins[bin_label]
    if len(bin_rows) < 200:
        print(f'\n{bin_label}: only {len(bin_rows)} rows — skipping')
        continue
    
    print(f'\n{"="*60}')
    print(f'BIN: {bin_label} ({len(bin_rows):,} rows)')
    print(f'{"="*60}')
    
    # Baseline (mean predictor)
    try:
        split_params = {'method': 'rolling', 'n_windows': SHARED['n_windows'],
                       'test_window_hours': SHARED['test_window_hours'],
                       'training_lookback_days': SHARED['training_lookback_days']}
        metric_defs = [{'name': 'mae', 'target': 'runtime_seconds'},
                      {'name': 'rmse', 'target': 'runtime_seconds'}]
        m, p, _ = run_rolling_baseline(bin_rows, split=split_params, metric_defs=metric_defs)
        results[(bin_label, 'Baseline')] = {'mae': m['mae'], 'rmse': m['rmse'],
                                            'scored': p['summary']['rows_scored']}
        print(f'  Baseline: MAE={m["mae"]:,.0f}s, scored={p["summary"]["rows_scored"]:,}')
    except Exception as e:
        print(f'  Baseline: FAILED — {e}')
    
    # All other models
    for model_name, model_factory in MODELS.items():
        try:
            model = model_factory()
            payload = model.evaluate(bin_rows)
            scored = payload['summary']['rows_scored']
            if scored > 0:
                results[(bin_label, model_name)] = {
                    'mae': payload['mae'], 'rmse': payload['rmse'], 'scored': scored}
                print(f'  {model_name}: MAE={payload["mae"]:,.0f}s, scored={scored:,}')
            else:
                print(f'  {model_name}: 0 scored')
        except Exception as e:
            print(f'  {model_name}: FAILED — {e}')

## 5. Results table

In [ ]:
print('=' * 80)
print('RESULTS: BEST MODEL PER BIN')
print('=' * 80)

all_models = ['Baseline', 'Random Forest', 'XGBoost Adjusted', 'XGBoost DART', 'LightGBM', 'TF-IDF kNN']

# Build results table
print(f'\n{"Bin":<10}', end='')
for m in all_models:
    print(f' {m:>14}', end='')
print(f' {"BEST":>14}')
print('-' * (10 + 15 * (len(all_models) + 1)))

best_per_bin = {}
for bl in BIN_LABELS:
    print(f'{bl:<10}', end='')
    bin_maes = {}
    for m in all_models:
        r = results.get((bl, m))
        if r:
            print(f' {r["mae"]:>12,.0f}s', end='')
            bin_maes[m] = r['mae']
        else:
            print(f' {"—":>14}', end='')
    if bin_maes:
        best = min(bin_maes, key=bin_maes.get)
        best_per_bin[bl] = best
        print(f' {best:>14}')
    else:
        print(f' {"—":>14}')

print(f'\n\nSummary — best model per bin:')
for bl, model in best_per_bin.items():
    mae = results[(bl, model)]['mae']
    print(f'  {bl:<10} → {model} (MAE={mae:,.0f}s)')

In [ ]:
# Heatmap
active_bins = [bl for bl in BIN_LABELS if any(results.get((bl, m)) for m in all_models)]
mae_matrix = []
for bl in active_bins:
    row = []
    for m in all_models:
        r = results.get((bl, m))
        row.append(r['mae'] if r else np.nan)
    mae_matrix.append(row)

mae_matrix = np.array(mae_matrix)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(mae_matrix, cmap='RdYlGn_r', aspect='auto')

ax.set_xticks(range(len(all_models)))
ax.set_xticklabels(all_models, rotation=30, ha='right')
ax.set_yticks(range(len(active_bins)))
ax.set_yticklabels(active_bins)

# Annotate cells
for i in range(len(active_bins)):
    for j in range(len(all_models)):
        val = mae_matrix[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:,.0f}', ha='center', va='center', fontsize=9)

ax.set_title('MAE by Model × Wallclock Bin (lower = better, green = best)')
plt.colorbar(im, label='MAE (seconds)')
plt.tight_layout()
plt.show()

## 6. Conclusions

In [ ]:
print('CONCLUSIONS')
print('=' * 60)

# Is one model best everywhere?
unique_winners = set(best_per_bin.values())
if len(unique_winners) == 1:
    print(f'\nOne model wins every bin: {list(unique_winners)[0]}')
    print('A single model type is sufficient — no need for per-bin model selection.')
else:
    print(f'\nDifferent models win different bins:')
    for bl, model in best_per_bin.items():
        print(f'  {bl}: {model}')
    print(f'\nUsing per-bin optimal models would further improve the MoE approach.')
    
    # How much would per-bin-optimal improve over using one model everywhere?
    # Compare best-everywhere vs per-bin-optimal
    from collections import Counter
    model_wins = Counter(best_per_bin.values())
    most_common = model_wins.most_common(1)[0][0]
    
    single_total_mae = 0
    single_total_scored = 0
    optimal_total_mae = 0
    optimal_total_scored = 0
    
    for bl in active_bins:
        # Single model (most common winner)
        r_single = results.get((bl, most_common))
        if r_single:
            single_total_mae += r_single['mae'] * r_single['scored']
            single_total_scored += r_single['scored']
        # Per-bin optimal
        best_model = best_per_bin.get(bl)
        if best_model:
            r_opt = results.get((bl, best_model))
            if r_opt:
                optimal_total_mae += r_opt['mae'] * r_opt['scored']
                optimal_total_scored += r_opt['scored']
    
    if single_total_scored > 0 and optimal_total_scored > 0:
        single_avg = single_total_mae / single_total_scored
        optimal_avg = optimal_total_mae / optimal_total_scored
        diff = (optimal_avg - single_avg) / single_avg * 100
        print(f'\n  {most_common} everywhere: weighted MAE = {single_avg:,.0f}s')
        print(f'  Per-bin optimal:          weighted MAE = {optimal_avg:,.0f}s')
        print(f'  Improvement from per-bin model selection: {diff:+.1f}%')